In [0]:
import pandas as pd
from pathlib import Path

LAKEHOUSE_PATH = "/Volumes/analytics/digital_twin/data"
SILVER_PATH = f"{LAKEHOUSE_PATH}/silver"
GOLD_PATH = f"{LAKEHOUSE_PATH}/gold"

input_file = f"{SILVER_PATH}/dados_refrigeracao_enriched.csv"
output_file = f"{GOLD_PATH}/dados_refrigeracao_gold.parquet"

# Leitura
df = pd.read_csv(input_file)
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Usar apenas dados fisicamente válidos
df_valid = df.loc[~df["flag_invalido"]].copy()
df_valid = df_valid.sort_values("timestamp").reset_index(drop=True)

# Métrica de variabilidade local
df_valid["cop_std_rolling"] = df_valid["cop"].rolling(window=50, min_periods=10).std()

# Classificação de regime
df_valid["regime_operacao"] = pd.cut(
    df_valid["cop"],
    bins=[0, 2.5, 3.2, 5],
    labels=["baixo", "medio", "alto"],
    include_lowest=True
)

# Alerta operacional
df_valid["status_alerta"] = df_valid["cop"] < 2.5

# Evento anômalo: combinação de baixa eficiência, instabilidade ou vazão baixa
df_valid["evento_anomalo"] = (
    (df_valid["cop"] < 2.3) |
    (df_valid["cop_std_rolling"] > 0.35) |
    (df_valid["vazao_m3_h"] < 70)
)

# Seleção final
cols_final = [
    "timestamp",
    "cop",
    "vazao_m3_h",
    "dT_c",
    "potencia_kw",
    "pressao_oleo_bar",
    "nivel_tanque_pct",
    "regime_operacao",
    "status_alerta",
    "evento_anomalo"
]

df_gold = df_valid[cols_final].copy()

# Garantir pasta
Path(GOLD_PATH).mkdir(parents=True, exist_ok=True)

# Salvar em parquet usando Spark para garantir compatibilidade com Databricks SQL
spark_df_gold = spark.createDataFrame(df_gold)
spark_df_gold = spark_df_gold.withColumn("timestamp", spark_df_gold["timestamp"].cast("timestamp"))
spark_df_gold.write.mode("overwrite").parquet(output_file)

df_gold.head()

In [0]:
print(df_gold["regime_operacao"].value_counts(dropna=False))
print(df_gold["status_alerta"].value_counts(dropna=False))
print(df_gold["evento_anomalo"].value_counts(dropna=False))